# 01 기초: 물리 제약이 있는 위성-작업 스케줄링

AEOS(Agile Earth Observation Satellite, 민첩 지구관측 위성)의 time window, 자세 전환, FOV(Field of View, 시야), 에너지 제약을 작은 문제로 구현한다. 논문의 대규모 Basilisk simulator를 재현하는 것이 아니라, 왜 거리만 가까운 작업을 선택하면 실패하는지 이해하는 toy reproduction이다.

In [1]:
from dataclasses import dataclass
from math import hypot

@dataclass(frozen=True)
class Satellite:
    name: str
    x: float
    y: float
    attitude_deg: float
    slew_rate_deg_s: float
    energy_wh: float
    sensor_w: float

@dataclass(frozen=True)
class Task:
    name: str
    x: float
    y: float
    look_angle_deg: float
    release_s: float
    due_s: float
    duration_s: float

satellites = [
    Satellite('S1', 0, 0, 0, 2.0, 12.0, 8.0),
    Satellite('S2', 10, 2, 70, 5.0, 5.0, 6.0),
]
tasks = [
    Task('T1', 1, 1, 80, 0, 50, 10),
    Task('T2', 8, 2, 60, 5, 30, 20),
    Task('T3', 5, 5, 10, 0, 12, 8),
]
len(satellites), len(tasks)

(2, 3)

## 실행 가능성 함수

필요 자세 전환 시간은 각도 차이를 slew rate(초당 회전 가능 각도)로 나눈다. 관측 시작은 release time과 전환 완료 시각 중 늦은 값이다. 배터리 에너지는 단순히 `sensor power × duration / 3600`으로 계산한다.

In [2]:
def pair_features(sat, task, now_s=0.0):
    distance = hypot(sat.x-task.x, sat.y-task.y)
    angle_delta = abs((task.look_angle_deg-sat.attitude_deg+180)%360-180)
    control_s = angle_delta / sat.slew_rate_deg_s
    start_s = max(task.release_s, now_s + control_s)
    finish_s = start_s + task.duration_s
    energy_wh = sat.sensor_w * task.duration_s / 3600
    feasible = finish_s <= task.due_s and energy_wh <= sat.energy_wh
    return {'distance': round(distance,2), 'control_s': round(control_s,2),
            'finish_s': round(finish_s,2), 'energy_wh': round(energy_wh,4),
            'feasible': feasible}

matrix = {(s.name,t.name): pair_features(s,t) for s in satellites for t in tasks}
matrix

{('S1', 'T1'): {'distance': 1.41,
  'control_s': 40.0,
  'finish_s': 50.0,
  'energy_wh': 0.0222,
  'feasible': True},
 ('S1', 'T2'): {'distance': 8.25,
  'control_s': 30.0,
  'finish_s': 50.0,
  'energy_wh': 0.0444,
  'feasible': False},
 ('S1', 'T3'): {'distance': 7.07,
  'control_s': 5.0,
  'finish_s': 13.0,
  'energy_wh': 0.0178,
  'feasible': False},
 ('S2', 'T1'): {'distance': 9.06,
  'control_s': 2.0,
  'finish_s': 12.0,
  'energy_wh': 0.0167,
  'feasible': True},
 ('S2', 'T2'): {'distance': 2.0,
  'control_s': 2.0,
  'finish_s': 25,
  'energy_wh': 0.0333,
  'feasible': True},
 ('S2', 'T3'): {'distance': 5.83,
  'control_s': 12.0,
  'finish_s': 20.0,
  'energy_wh': 0.0133,
  'feasible': False}}

## 거리 기반 선택과 제약 기반 선택 비교

거리만 최소화한 후보가 due time을 놓칠 수 있다. 실제 AEOS-Bench는 훨씬 많은 동역학과 continuity 제약을 simulator에서 검사한다.

In [3]:
def choose(sat, constraint_aware):
    candidates = [(pair_features(sat,t), t) for t in tasks]
    if constraint_aware:
        candidates = [(f,t) for f,t in candidates if f['feasible']]
    return min(candidates, key=lambda item: item[0]['distance'])[1].name if candidates else None

comparison = {s.name: {'distance_only': choose(s, False),
                       'constraint_aware': choose(s, True)} for s in satellites}
comparison

{'S1': {'distance_only': 'T1', 'constraint_aware': 'T1'},
 'S2': {'distance_only': 'T2', 'constraint_aware': 'T2'}}

In [4]:
# 경계 검증: 선택된 constraint-aware 쌍은 모두 실행 가능해야 한다.
for sat in satellites:
    selected = choose(sat, True)
    if selected is not None:
        assert matrix[(sat.name, selected)]['feasible']
print('검증 통과:', comparison)

검증 통과: {'S1': {'distance_only': 'T1', 'constraint_aware': 'T1'}, 'S2': {'distance_only': 'T2', 'constraint_aware': 'T2'}}


## 확장 과제

1. 두 위성이 같은 작업을 선택하지 못하도록 bipartite matching을 구현한다.
2. 자세 전환 후 satellite state를 갱신해 여러 timestep을 시뮬레이션한다.
3. 태양전지 충전, reaction-wheel momentum, target visibility interval을 추가한다.